# TICKET CLASSIFIER FUNCTION - USING GROQ API

In [1]:
import json
from groq import Groq
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np

In [5]:
# Load API key
load_dotenv()
api_key = os.getenv('GROQ_API_KEY')
client = Groq(api_key=api_key)

def classify_ticket_with_groq(subject, body, language='en', max_retries=3):
    """
    Classify a support ticket by priority using Groq/Llama API.
    
    Parameters:
    -----------
    subject : str
        The ticket subject/title
    body : str
        The ticket description/body (main text)
    language : str
        Language of the ticket ('en' for English, 'de' for German)
    max_retries : int
        How many times to retry if API fails
    
    Returns:
    --------
    dict : Classification result with keys:
        - 'priority': 'low', 'medium', or 'high'
        - 'confidence': confidence score (0-100)
        - 'reasoning': explanation for the decision
        - 'key_factors': list of factors that influenced decision
        - 'success': True/False (was the call successful?)
    """
    
    # Create the prompt that will be sent to Llama
    prompt = f"""You are an expert customer support triage specialist. Your job is to classify 
support tickets by priority to ensure critical issues are handled first.

PRIORITY LEVELS:
- LOW: Single user, non-critical issue, can wait 3-5 days
  Examples: Feature request, documentation question, feedback, general inquiry
  
- MEDIUM: Multiple users affected OR important issue, can wait 1 day  
  Examples: Feature not working properly, performance degradation, billing question
  
- HIGH: Critical or urgent, needs immediate attention (within 2 hours)
  Examples: System down, data loss, security issue, many users affected, revenue impact

FACTORS TO CONSIDER:
1. Number of users affected (1 = low, few = medium, many/all = high)
2. Business impact (no impact = low, some impact = medium, critical = high)
3. Is there a deadline/urgency? (yes = higher priority)
4. Is there urgent language? (frustrated, critical, ASAP, emergency = higher)
5. Is a core system/service down? (yes = high)

TICKET LANGUAGE: {language}

TICKET:
Subject: {subject}
Body: {body}

Classify this ticket by analyzing the above factors.

Respond ONLY with this exact JSON format (no extra text):
{{
    "priority": "[low/medium/high]",
    "confidence": [0-100],
    "reasoning": "[1-2 sentence explanation]",
    "key_factors": ["factor1", "factor2", "factor3"]
}}"""

    # Try to classify (with retry logic)
    for attempt in range(max_retries):
        try:
            # Send prompt to Groq/Llama and get response
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                max_tokens=300,
                messages=[
                    {"role": "user", "content": prompt}
                ]
            )
            
            # Extract the text response
            response_text = response.choices[0].message.content.strip()
            
            # Parse the JSON response
            result = json.loads(response_text)
            
            # Add success flag
            result['success'] = True
            return result
            
        except json.JSONDecodeError as e:
            # If response wasn't valid JSON, retry
            if attempt < max_retries - 1:
                continue
            else:
                return {
                    'priority': 'unknown',
                    'confidence': 0,
                    'reasoning': f'Failed to parse response',
                    'key_factors': [],
                    'success': False
                }
                
        except Exception as e:
            # If API call failed, retry
            if attempt < max_retries - 1:
                continue
            else:
                return {
                    'priority': 'unknown',
                    'confidence': 0,
                    'reasoning': f'API error',
                    'key_factors': [],
                    'success': False
                }

print(" Helper function 'classify_ticket_with_groq' created successfully!")
print("\nThis function will:")
print("  1. Take ticket subject and body as input")
print("  2. Send to Groq/Llama API with detailed prompt")
print("  3. Parse the structured JSON response")
print("  4. Return priority classification")
print("  5. Handle errors gracefully with retries")

 Helper function 'classify_ticket_with_groq' created successfully!

This function will:
  1. Take ticket subject and body as input
  2. Send to Groq/Llama API with detailed prompt
  3. Parse the structured JSON response
  4. Return priority classification
  5. Handle errors gracefully with retries


### TEST THE CLASSIFIER ON REAL TICKETS

In [18]:
print("=" * 80)
print("TESTING CLASSIFIER ON REAL TICKETS")
print("=" * 80)

# Select sample tickets from each priority level
test_tickets = []

# Get 2 LOW priority tickets
low_tickets = df[df['priority'] == 'low'].sample(min(2, len(df[df['priority'] == 'low'])))
for idx, row in low_tickets.iterrows():
    test_tickets.append({
        'actual_priority': 'low',
        'subject': row['subject'],
        'body': row['body'],
        'language': row['language']
    })

# Get 2 MEDIUM priority tickets
medium_tickets = df[df['priority'] == 'medium'].sample(min(2, len(df[df['priority'] == 'medium'])))
for idx, row in medium_tickets.iterrows():
    test_tickets.append({
        'actual_priority': 'medium',
        'subject': row['subject'],
        'body': row['body'],
        'language': row['language']
    })

# Get 2 HIGH priority tickets
high_tickets = df[df['priority'] == 'high'].sample(min(2, len(df[df['priority'] == 'high'])))
for idx, row in high_tickets.iterrows():
    test_tickets.append({
        'actual_priority': 'high',
        'subject': row['subject'],
        'body': row['body'],
        'language': row['language']
    })

print(f"\n Testing classifier on {len(test_tickets)} sample tickets")
print(f"   {len(low_tickets)} LOW priority")
print(f"   {len(medium_tickets)} MEDIUM priority")
print(f"   {len(high_tickets)} HIGH priority")

print("\n" + "=" * 80)
print("PROCESSING TICKETS...")
print("=" * 80)

# Store results
results = []

# Process each test ticket
for i, ticket in enumerate(test_tickets, 1):
    subject_preview = ticket['subject'][:40] if ticket['subject'] else "No subject"
    print(f"\n[{i}/{len(test_tickets)}] {subject_preview}...")
    
    # Classify the ticket
    classification = classify_ticket_with_groq(
        subject=ticket['subject'],
        body=ticket['body'],
        language=ticket['language']
    )
    
    # Add actual priority to result
    classification['actual_priority'] = ticket['actual_priority']
    results.append(classification)
    
    # Show result
    if classification['success']:
        match = "✅" if classification['priority'] == ticket['actual_priority'] else "❌"
        print(f"   {match} Predicted: {classification['priority']:6s} | Actual: {ticket['actual_priority']:6s}")
        print(f"      Confidence: {classification['confidence']}%")
        print(f"      Reasoning: {classification['reasoning'][:70]}...")
    else:
        print(f"   ❌ Failed: {classification['reasoning']}")

print("\n" + "=" * 80)
print("TEST RESULTS SUMMARY")
print("=" * 80)

# Calculate accuracy
successful = sum(1 for r in results if r['success'])
correct = sum(1 for r in results if r['success'] and r['priority'] == r['actual_priority'])

if successful > 0:
    accuracy = (correct / successful) * 100
    print(f"\n✅ Successful classifications: {successful}/{len(results)}")
    print(f"✅ Correct predictions: {correct}/{successful}")
    print(f"✅ Accuracy on test set: {accuracy:.1f}%")
    print(f"✅ Average confidence: {np.mean([r['confidence'] for r in results if r['success']]):.1f}%")
else:
    print("❌ No successful classifications")

# Show breakdown by priority level
print("\n" + "-" * 80)
print("BREAKDOWN BY PRIORITY LEVEL:")
print("-" * 80)

for priority in ['low', 'medium', 'high']:
    priority_results = [r for r in results if r['actual_priority'] == priority and r['success']]
    if priority_results:
        correct_for_priority = sum(1 for r in priority_results if r['priority'] == priority)
        accuracy_for_priority = (correct_for_priority / len(priority_results)) * 100
        print(f"{priority.upper():7s}: {correct_for_priority}/{len(priority_results)} correct ({accuracy_for_priority:.0f}%)")

print("\n" + "=" * 80)

TESTING CLASSIFIER ON REAL TICKETS

 Testing classifier on 6 sample tickets
   2 LOW priority
   2 MEDIUM priority
   2 HIGH priority

PROCESSING TICKETS...

[1/6] Bekanntgabe Verletzung der medizinischen...
   ❌ Predicted: high   | Actual: low   
      Confidence: 95%
      Reasoning: The ticket reports a medical data security breach, indicating a critic...

[2/6] Support for Marketing Agency Strategies...
   ✅ Predicted: low    | Actual: low   
      Confidence: 90%
      Reasoning: The ticket describes a non-critical issue with a marketing agency's di...

[3/6] Request for Assistance with Service Inte...
   ❌ Predicted: high   | Actual: medium
      Confidence: 95%
      Reasoning: The ticket mentions a service interruption and issues with the medical...

[4/6] Billing Discrepancy Inquiry...
   ✅ Predicted: medium | Actual: medium
      Confidence: 80%
      Reasoning: The ticket reports a billing discrepancy affecting multiple software p...

[5/6] Sicherheitsrisiko im Gesundheitswe